In [ ]:
# Ultralytics YOLO 학습과 추론 라이브러리를 설치합니다.
!pip install ultralytics

In [ ]:
# PyTorch와 이미지/오디오 관련 패키지를 설치합니다.
!pip install torch torchvision torchaudio -y


Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: -y


In [ ]:
# CUDA 11.8 환경에 맞는 PyTorch 패키지를 설치합니다.
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached torch-2.4.0%2Bcu118-cp311-cp311-win_amd64.whl (2692.5 MB)
  Using cached torchvision-0.19.0%2Bcu118-cp311-cp311-win_amd64.whl (5.0 MB)

  Attempting uninstall: torch

    Found existing installation: torch 2.14.0

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
    Uninstalling torch-2.14.0:
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
      Successfully uninstalled torch-2.14.0
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.4.152 requires torch!=2.4.0,>=1.8.0; sys_platform == "win32", but you have torch 2.4.0+cu118 which is incompatible.


In [ ]:
# GPU 연산을 사용하기 위해 PyTorch를 불러옵니다.
import torch

# 현재 환경에서 CUDA GPU를 사용할 수 있는지 확인합니다.
print(torch.cuda.is_available())

True


In [ ]:
# 학습 데이터 분할에 사용할 scikit-learn을 설치합니다.
!pip install scikit-learn

In [ ]:
# 파일과 폴더를 다루기 위해 os를 불러옵니다.
import os
# 파일을 복사하거나 폴더를 삭제하기 위해 shutil을 불러옵니다.
import shutil
# 이미지 목록을 학습용과 검증용으로 나누기 위해 함수를 불러옵니다.
from sklearn.model_selection import train_test_split

# YOLO 모델을 불러옵니다.
from ultralytics import YOLO

WARNING Known issue with torch==2.4.0 on Windows with CPU, recommend upgrading to torch>=2.4.1 to resolve https://github.com/ultralytics/ultralytics/issues/15049


In [ ]:
# YOLO 형식으로 정리할 데이터셋 폴더의 경로를 지정합니다.
dataset_dir = r"C:/ai_project01/mask_yolo_dataset"

# 기존 데이터셋 폴더가 있으면 새로 만들기 위해 삭제합니다.
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)
    # 삭제가 완료되었음을 출력합니다.
    print(f"Deleted existing dataset directory: {dataset_dir}")

Deleted existing dataset directory: C:/ai_project01/mask_yolo_dataset


In [ ]:
# 학습 이미지가 저장될 폴더의 경로를 만듭니다.
image_train_dir = os.path.join(dataset_dir, "images/train")
# 검증 이미지가 저장될 폴더의 경로를 만듭니다.
image_val_dir = os.path.join(dataset_dir, "images/val")
# 학습 라벨이 저장될 폴더의 경로를 만듭니다.
label_train_dir = os.path.join(dataset_dir, "labels/train")
# 검증 라벨이 저장될 폴더의 경로를 만듭니다.
label_val_dir = os.path.join(dataset_dir, "labels/val")

In [ ]:
# 학습/검증 이미지와 라벨 폴더를 순서대로 생성합니다.
for d in [image_train_dir, image_val_dir, label_train_dir, label_val_dir]:
    # 폴더가 없어도 만들고, 이미 있으면 오류 없이 넘어갑니다.
    os.makedirs(d, exist_ok=True)
    # 생성한 폴더 경로를 출력합니다.
    print(f"Created directory: {d}")

Created directory: C:/ai_project01/mask_yolo_dataset\images/train
Created directory: C:/ai_project01/mask_yolo_dataset\images/val
Created directory: C:/ai_project01/mask_yolo_dataset\labels/train
Created directory: C:/ai_project01/mask_yolo_dataset\labels/val


In [ ]:
# 원본 이미지가 저장된 폴더의 경로를 지정합니다.
image_src_dir = r"C:/ai_project01/mask_images"
# 원본 YOLO 라벨이 저장된 폴더의 경로를 지정합니다.
label_src_dir = r"C:/ai_project01/labels"

In [ ]:
# 전체 이미지 파일의 경로를 저장할 빈 목록을 만듭니다.
image_files = []

# 마스크 착용과 미착용 폴더를 차례로 확인합니다.
for subdir in ['mask_on', 'no_mask']:
    # 현재 클래스 폴더의 전체 경로를 만듭니다.
    subdir_path = os.path.join(image_src_dir, subdir)
    # 현재 폴더의 파일 이름을 하나씩 확인합니다.
    for f in os.listdir(subdir_path):
        # 이미지 확장자를 가진 파일만 선택합니다.
        if f.endswith(('.jpg', '.jpeg', '.png')):
            # 파일 이름과 폴더를 합쳐 이미지 전체 경로를 만듭니다.
            full_file_path = os.path.join(subdir_path, f)
            # 이미지 경로를 전체 목록에 추가합니다.
            image_files.append(full_file_path)

In [ ]:
# 전체 이미지의 80%는 학습용, 20%는 검증용으로 나눕니다.
# random_state를 고정해 실행할 때마다 같은 분할 결과를 얻습니다.
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

In [ ]:
# 이미지 파일과 대응하는 라벨 파일을 목적지 폴더로 복사하는 함수를 정의합니다.
def move_files(files, image_dest, label_dest):
    # 전달받은 이미지 목록을 하나씩 처리합니다.
    for img_path in files:
        # 이미지 경로에서 파일 이름만 추출합니다.
        filename = os.path.basename(img_path)
        # 이미지와 같은 이름을 가진 TXT 라벨 파일 이름을 만듭니다.
        label_name = os.path.splitext(filename)[0] + ".txt"
        # 원본 라벨 파일의 전체 경로를 만듭니다.
        label_path = os.path.join(label_src_dir, label_name)

        # 대응하는 라벨 파일이 실제로 존재하는지 확인합니다.
        if os.path.exists(label_path):
            # 이미지를 목적지의 학습 또는 검증 폴더로 복사합니다.
            shutil.copy2(img_path, image_dest)
            # 라벨 파일도 같은 구분의 라벨 폴더로 복사합니다.
            shutil.copy2(label_path, label_dest)
        else:
            # 라벨이 없으면 해당 이미지 경로를 알려줍니다.
            print(f"Label file not found for image: {img_path}")

In [ ]:
# 학습용 이미지와 라벨을 각각의 학습 폴더로 복사합니다.
move_files(train_files, image_train_dir, label_train_dir)
# 검증용 이미지와 라벨을 각각의 검증 폴더로 복사합니다.
move_files(val_files, image_val_dir, label_val_dir)
# 데이터셋 복사가 완료되었음을 출력합니다.
print("Files moved successfully.")

Files moved successfully.


In [ ]:
# YOLO가 읽을 데이터 설정 파일의 경로를 만듭니다.
data_yaml_path = os.path.join(dataset_dir, "data.yaml")

# 학습/검증 이미지 경로와 클래스 이름을 YAML 형식으로 작성합니다.
yaml_content = f"""
train: C:/ai_project01/mask_yolo_dataset/images/train
val: C:/ai_project01/mask_yolo_dataset/images/val

names:
  0: mask_on
  1: no_mask

nc: 2
"""
# data.yaml 파일을 쓰기 모드로 엽니다.
with open(data_yaml_path, 'w') as file:
    # 앞뒤 불필요한 공백을 제거한 설정 내용을 파일에 저장합니다.
    file.write(yaml_content.strip())

# 생성된 YAML 파일의 경로를 출력합니다.
print(f"data.yaml file created at: {data_yaml_path}")

data.yaml file created at: C:/ai_project01/mask_yolo_dataset\data.yaml


In [ ]:
# 사전 학습된 YOLOv8m 모델을 불러옵니다.
model = YOLO("yolov8m.pt")

# 준비한 데이터셋으로 YOLO 모델 학습을 시작합니다.
results = model.train(
    # 학습/검증 이미지 경로와 클래스 정보를 담은 YAML 파일입니다.
    data=data_yaml_path,
    # 학습 결과 가중치와 로그를 저장합니다.
    save=True,
    # 전체 학습 반복 횟수입니다.
    epochs=1000,
    # 입력 이미지의 한 변 크기를 640픽셀로 조정합니다.
    imgsz=640,
    # 한 번에 처리할 이미지 수입니다.
    batch=16,
    # 결과 폴더에 사용할 학습 이름입니다.
    name='mask_detection',
    # 학습 결과를 저장할 상위 폴더입니다.
    project=r"C:/ai_project01/results",
    # 같은 이름의 결과 폴더가 있어도 덮어쓸 수 있게 합니다.
    exist_ok=True,
    # 30번 연속으로 개선이 없으면 조기 종료합니다.
    patience=30
)
# 학습이 끝났고 결과가 저장되었음을 출력합니다.
print("Training completed. Model saved in the results directory.")

Ultralytics 8.4.152  Python-3.11.16 torch-2.4.0+cu118 CUDA:0 (RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/ai_project01/mask_yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1000, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mask_detection, nbs=64, nms=None,